In [28]:
from __future__ import annotations

import operator
import os
import re
from datetime import date, datetime, timedelta
from pathlib import Path
from typing import TypedDict, List, Optional, Literal, Annotated

from pydantic import BaseModel, Field

from langgraph.graph import StateGraph, START, END
from langgraph.types import Send

from langchain_openrouter import ChatOpenRouter

from langchain_core.messages import SystemMessage, HumanMessage
from dotenv import load_dotenv
from langchain_community.tools.tavily_search import TavilySearchResults

In [29]:
import sys
import os
import httpx
import anyio
import ipykernel.iostream
from dotenv import load_dotenv

# Fix Jupyter OutStream fileno error for Windows MCP subprocesses
ipykernel.iostream.OutStream.fileno = lambda self: sys.__stderr__.fileno()
sys.stderr = sys.__stderr__

load_dotenv()

import langchain
langchain.debug = False

from langchain_mcp_adapters.client import MultiServerMCPClient
from mcp.client.session import ClientSession
from mcp.types import JSONRPCMessage

# Add Streamable HTTP transport support to MultiServerMCPClient
async def _connect_via_http(self, server_name: str, *, url: str, headers: dict = None, **kwargs):
    req_headers = dict(headers or {})
    req_headers['Content-Type'] = 'application/json'
    read_stream_writer, read_stream = anyio.create_memory_object_stream(0)
    write_stream, write_stream_reader = anyio.create_memory_object_stream(0)
    session_id = None

    client = await self.exit_stack.enter_async_context(httpx.AsyncClient(headers=req_headers, timeout=30.0))
    tg = await self.exit_stack.enter_async_context(anyio.create_task_group())

    async def reader():
        nonlocal session_id
        async with write_stream_reader:
            async for message in write_stream_reader:
                post_headers = dict(req_headers)
                if session_id:
                    post_headers['Mcp-Session-Id'] = session_id
                resp = await client.post(url, json=message.model_dump(by_alias=True, mode='json', exclude_none=True), headers=post_headers)
                if 'mcp-session-id' in resp.headers:
                    session_id = resp.headers['mcp-session-id']
                for line in resp.text.splitlines():
                    if line.startswith('data: '):
                        data_str = line[6:].strip()
                        if data_str:
                            msg = JSONRPCMessage.model_validate_json(data_str)
                            await read_stream_writer.send(msg)

    tg.start_soon(reader)
    session = await self.exit_stack.enter_async_context(ClientSession(read_stream, write_stream))
    await self._initialize_session_and_load_tools(server_name, session)

MultiServerMCPClient.connect_to_server_via_http = _connect_via_http

async def _patched_aenter(self):
    try:
        connections = self.connections or {}
        for server_name, connection in connections.items():
            connection_dict = connection.copy()
            transport = connection_dict.pop('transport')
            if transport == 'http':
                await self.connect_to_server_via_http(server_name, **connection_dict)
            elif transport == 'stdio':
                await self.connect_to_server_via_stdio(server_name, **connection_dict)
            elif transport == 'sse':
                await self.connect_to_server_via_sse(server_name, **connection_dict)
        return self
    except Exception:
        await self.exit_stack.aclose()
        raise

MultiServerMCPClient.__aenter__ = _patched_aenter

github_token = os.environ.get('GITHUB_ACCESS_TOKEN')

# Initialize MCP client with GitHub Copilot HTTP MCP server
mcp_client = MultiServerMCPClient(
    {
        "github": {
            "transport": "http",
            "url": "https://api.githubcopilot.com/mcp",
            "headers": {
                "Authorization": f"Bearer {github_token}"
            },
        }
    }
)

await mcp_client.__aenter__()
mcp_tools = mcp_client.get_tools()

print(f"\n✅ Successfully loaded {len(mcp_tools)} total MCP tools:")



✅ Successfully loaded 44 total MCP tools:


In [30]:
for tool in mcp_tools:
    print(f" - Tool: {tool.name}")  
    print(f"   Description: {tool.description}")
    print(f"   Input Schema: {tool.input_schema}")

 - Tool: add_comment_to_pending_review
   Description: Add review comment to the requester's latest pending pull request review. A pending review needs to already exist to call this (check with the user if not sure).
   Input Schema: <class 'langchain_core.tools.structured.add_comment_to_pending_review_input'>
 - Tool: add_issue_comment
   Description: Add a comment and/or reaction to a specific issue or issue comment in a GitHub repository. Use this tool with pull requests as well (in this case pass pull request number as issue_number), but only if user is not asking specifically to add or react to review comments. At least one of body or reaction is required.
   Input Schema: <class 'langchain_core.tools.structured.add_issue_comment_input'>
 - Tool: add_reply_to_pull_request_comment
   Description: Add a reply and/or reaction to an existing pull request comment. This can create a new comment linked as a reply to the specified comment, add an emoji reaction to the specified comment, o

In [31]:
get_file_contents_tool = next(
    tool for tool in mcp_tools
    if tool.name == "get_file_contents"
)

get_file_contents_tool.args_schema

{'type': 'object',
 'properties': {'fields': {'type': 'array',
   'items': {'type': 'string',
    'enum': ['type',
     'name',
     'path',
     'size',
     'sha',
     'url',
     'git_url',
     'html_url',
     'download_url']},
   'description': "Subset of fields to return for each entry when the path is a directory. If omitted, all fields are returned. Ignored when the path is a single file. Use this to reduce response size when listing directories and you only need specific fields, e.g. just 'name' and 'type'."},
  'owner': {'type': 'string',
   'description': 'Repository owner (username or organization)',
   'x-mcp-header': 'owner'},
  'path': {'type': 'string',
   'description': 'Path to file/directory',
   'default': '/'},
  'ref': {'type': 'string',
   'description': 'Accepts optional git refs such as `refs/tags/{tag}`, `refs/heads/{branch}` or `refs/pull/{pr_number}/head`'},
  'repo': {'type': 'string',
   'description': 'Repository name',
   'x-mcp-header': 'repo'},
  'sh

In [37]:
result = await get_file_contents_tool.ainvoke({
    "owner": "github",
    "repo": "github-mcp-server",
    "path": "README.md"
})

result

'successfully downloaded text file (SHA: 6585ab30f6d30c81970084cd8a15b750cc57b8f5)'

In [40]:
print("TYPE:")
print(type(result))

print("\nDIR:")
print(dir(result))

print("\nREPR:")
print(repr(result))

TYPE:
<class 'str'>

DIR:
['__add__', '__class__', '__contains__', '__delattr__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getitem__', '__getnewargs__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__iter__', '__le__', '__len__', '__lt__', '__mod__', '__mul__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__rmod__', '__rmul__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', 'capitalize', 'casefold', 'center', 'count', 'encode', 'endswith', 'expandtabs', 'find', 'format', 'format_map', 'index', 'isalnum', 'isalpha', 'isascii', 'isdecimal', 'isdigit', 'isidentifier', 'islower', 'isnumeric', 'isprintable', 'isspace', 'istitle', 'isupper', 'join', 'ljust', 'lower', 'lstrip', 'maketrans', 'partition', 'removeprefix', 'removesuffix', 'replace', 'rfind', 'rindex', 'rjust', 'rpartition', 'rsplit', 'rstrip', 'split', 'splitlines', 'startswith', 'strip', 'swapcase', 'title', 'translate', 'upper', 'z